# Milestone - Unit 3: a model of the car that predicts

Unit 3 asked one question: **where will the car be in one second?**
This milestone pins the answer the submission now carries.
`agents/unit-03.py` is the starter with Unit 2's track geometry and a kinematic-bicycle model added,
and a `predict(state, actions, dt=0.05)` that rolls the model forward one RK4 step per action.
It does not change the driving: on racekit's kinematic car the starter neither wobbles nor cuts corners,
so this unit adds an instrument, not a controller.

The notebook asserts, rather than displays, each fact the unit taught:

- **G3.1** the steady circle: RK4 stays on R = L / tan(s·δ_max) within 1e-6 m;
  Euler's distance from the true centre is off by about h·v/2 = 0.25 m, bounded, the same in the first and the last 200 steps;
- **G3.2** prediction from telemetry: replaying the recorded actions for 1 s from 180 start rows of the deterministic `monza-like` race,
  RK4 at one step per tick reproduces racekit's position within 1e-6 m, and Euler is about 0.378 m off at one step per tick and about 0.038 m at ten substeps (first order);
- **G3.3** the convergence orders: error ratios per halving of h are about 2 for Euler and about 16 for RK4;
- **G3.4** the submission races exactly like the starter (223.50 / 412.75 / 882.55 s, clean), passes `check_submission`,
  and its `predict()` reproduces the full recorded state (x, y, ψ, v) of every G3.2 window.

The model's parameters are racekit's **kinematic car's**: a model, not the truth (Unit 8 re-identifies them from data).

## 1. The model, written independently of the submission

The milestone runs from `projects/racekit/milestones/`, so the submission is one directory up.
The model here is a NumPy restatement of the unit's equations; G3.4 checks the submission's own scalar version against racekit.

In [ ]:
import math
from pathlib import Path

import numpy as np
from racekit.cars import KinematicCar
from racekit.check_submission import check_submission
from racekit.runner import CONTROL_DT, load_submission, run_race, shim_installed

AGENT = Path("..") / "agents" / "unit-03.py"
STARTER = Path("..") / "agents" / "unit-01.py"
TOL = 1e-9  # race times are sums of 0.05 s ticks, so "exact" means to float rounding
PINNED_TIMES = {"oval": 223.50, "test-circuit": 412.75, "monza-like": 882.55}

WHEELBASE = 2.875
DELTA_MAX = math.radians(70.0)
A_THROTTLE, A_BRAKE, C_DRAG = 6.0, 8.0, 4.0e-4


def derivatives(state, action):
    _, _, psi, v = state
    throttle, steer, brake = action
    accel = A_THROTTLE * throttle - C_DRAG * v * abs(v)
    if v > 0.0:
        accel -= A_BRAKE * brake
    elif v < 0.0:
        accel += A_BRAKE * brake
    return np.array([v * math.cos(psi), v * math.sin(psi), v * math.tan(-steer * DELTA_MAX) / WHEELBASE, accel])


def euler_step(state, action, h):
    return state + h * derivatives(state, action)


def rk4_step(state, action, h):
    k1 = derivatives(state, action)
    k2 = derivatives(state + 0.5 * h * k1, action)
    k3 = derivatives(state + 0.5 * h * k2, action)
    k4 = derivatives(state + h * k3, action)
    return state + h / 6.0 * (k1 + 2.0 * k2 + 2.0 * k3 + k4)


def wrap_angle(angle):
    return (angle + math.pi) % (2.0 * math.pi) - math.pi

## 2. G3.1: the steady circle

Hold steer s at the drag-balancing throttle (c·v²/a_T) with v = 10 m/s, for 2000 steps of h = 0.05 s, from the origin heading +x.
A positive steer turns right, so the true centre is (0, −R).
Euler's trajectory is an exact regular polygon shifted by about h·v/2 along the initial heading, so its distance from the true centre swings between about R ± 0.25 m once per lap.
The statistic is therefore the **maximum** |dist − R| over a 200-step window, never the mean or the signed value.

In [ ]:
PINNED_RADII = {0.05: 47.006, 0.1: 23.415, 0.2: 11.531}


def circle_deviation(step, steer, h=0.05, steps=2000, v=10.0):
    radius = WHEELBASE / math.tan(steer * DELTA_MAX)
    action = (C_DRAG * v * v / A_THROTTLE, steer, 0.0)
    state = np.array([0.0, 0.0, 0.0, v])
    out = np.zeros(steps)
    for n in range(steps):
        state = step(state, action, h)
        out[n] = math.hypot(state[0], state[1] + radius) - radius
    return radius, out


for steer, pinned in PINNED_RADII.items():
    radius, rk4_dev = circle_deviation(rk4_step, steer)
    _, euler_dev = circle_deviation(euler_step, steer)
    first, last = np.max(np.abs(euler_dev[:200])), np.max(np.abs(euler_dev[-200:]))
    assert abs(radius - pinned) < 5e-4, radius
    assert np.max(np.abs(rk4_dev)) < 1e-6
    assert 0.245 <= first <= 0.255 and 0.245 <= last <= 0.255, (first, last)
    assert abs(first - last) <= 1e-3
    print(f"s = {steer}: R = {radius:.3f} m; RK4 max |dist - R| {np.max(np.abs(rk4_dev)):.1e} m; "
          f"Euler max |dist - R| first/last 200 steps {first:.4f} / {last:.4f} m")

## 3. G3.2: prediction from telemetry

The deterministic `monza-like` race of the submission itself.
Row i holds the action applied during tick i and the state after it,
so the state at row r + 20 is reached from row r by the actions of rows r + 1 … r + 20.
The start rows are 200, 297, … at a stride of 97: 180 windows of 1 s.

In [ ]:
monza = run_race(AGENT, "monza-like", car=KinematicCar(jitter=0.0))
telemetry = monza.telemetry
STARTS = np.arange(200, len(telemetry) - 20, 97)[:180]
assert len(STARTS) == 180 and STARTS[0] == 200 and STARTS[1] == 297


def start_state(row):
    return (telemetry["x"][row], telemetry["y"][row], telemetry["yaw"][row], telemetry["v"][row])


def window_actions(row):
    return [(telemetry["throttle"][i], telemetry["steer"][i], telemetry["brake"][i]) for i in range(row + 1, row + 21)]


def replay(step, substeps=1):
    h = CONTROL_DT / substeps
    out = np.zeros(len(STARTS))
    for m, row in enumerate(STARTS):
        state = np.array(start_state(row))
        for action in window_actions(row):
            for _ in range(substeps):
                state = step(state, action, h)
        end = row + 20
        out[m] = math.hypot(state[0] - telemetry["x"][end], state[1] - telemetry["y"][end])
    return out


rk4_errors = replay(rk4_step)
euler_errors = replay(euler_step)
euler10_errors = replay(euler_step, substeps=10)
print(f"RK4, 1 step per tick:   max {np.max(rk4_errors):.2e} m")
print(f"Euler, 1 step per tick: max {np.max(euler_errors):.4f} m, median {np.median(euler_errors):.1e} m")
print(f"Euler, 10 substeps:     max {np.max(euler10_errors):.5f} m")
assert np.max(rk4_errors) < 1e-6
assert abs(np.max(euler_errors) - 0.378) < 0.005
assert abs(np.max(euler10_errors) - 0.0378) < 0.0005
assert 9.9 < np.max(euler_errors) / np.max(euler10_errors) < 10.1  # first-order convergence
assert np.median(euler_errors) < 1e-5  # straights at the 19.49 m/s equilibrium: Euler is exact there

## 4. G3.3: the convergence orders

Throttle 0.6, steer 0.15, from v₀ = 5 m/s for T = 4 s; the reference is RK4 at h = 1e-4.
The order is read from the ratio of errors per halving of h (log₂ of it), not from a fitted line.

In [ ]:
def final_state(step, h, horizon=4.0, action=(0.6, 0.15, 0.0)):
    state = np.array([0.0, 0.0, 0.0, 5.0])
    for _ in range(round(horizon / h)):
        state = step(state, action, h)
    return state


reference = final_state(rk4_step, 1e-4)
STEPS = [0.4, 0.2, 0.1, 0.05]
for name, step, low, high in (("Euler", euler_step, 1.8, 2.2), ("RK4", rk4_step, 14.0, 18.0)):
    errors = np.array([math.hypot(*(final_state(step, h)[:2] - reference[:2])) for h in STEPS])
    ratios = errors[:-1] / errors[1:]
    print(f"{name:>5}: errors {[f'{e:.3g}' for e in errors]} m, ratios {np.round(ratios, 2)}, orders {np.round(np.log2(ratios), 2)}")
    assert np.all((low <= ratios) & (ratios <= high)), ratios

## 5. G3.4: the submission

Three facts.
It races exactly like the starter, at jitter 0, on all three tracks (and with identical telemetry on `oval`).
It passes `check_submission` (the isolated Python 3.8 test runs every file in `agents/` separately).
And its `predict()`, called on the 20 recorded actions of each G3.2 window, reproduces the recorded state at the window's end:
x and y within 1e-6 m, ψ within 1e-9 rad (compared wrapped, because racekit wraps the yaw and `predict` does not), and v within 1e-9 m/s.

In [ ]:
assert monza.finished and monza.collisions == 0 and abs(monza.total_time - PINNED_TIMES["monza-like"]) <= TOL
for track in ("oval", "test-circuit"):
    result = run_race(AGENT, track, car=KinematicCar(jitter=0.0))
    assert result.finished and result.collisions == 0, (track, result)
    assert abs(result.total_time - PINNED_TIMES[track]) <= TOL, (track, result.total_time)
    print(result)
    if track == "oval":
        starter_oval = run_race(STARTER, "oval", car=KinematicCar(jitter=0.0))
        for column in result.telemetry.dtype.names:  # identical telemetry, column by column
            assert np.max(np.abs(result.telemetry[column] - starter_oval.telemetry[column])) == 0, column
print(monza)

report = check_submission(AGENT)
print(report)
assert report.ok, report.findings

In [ ]:
with shim_installed():
    Agent, _ = load_submission(AGENT)
predict = Agent.predict  # the file's module-level predict(state, actions, dt=0.05)

worst = [0.0, 0.0, 0.0, 0.0]
for row in STARTS:
    x, y, psi, v = predict(start_state(row), window_actions(row))
    end = row + 20
    gaps = [
        abs(x - telemetry["x"][end]),
        abs(y - telemetry["y"][end]),
        abs(wrap_angle(psi - telemetry["yaw"][end])),
        abs(v - telemetry["v"][end]),
    ]
    worst = [max(old, new) for old, new in zip(worst, gaps)]
print(f"over {len(STARTS)} windows, max |error|: x {worst[0]:.1e} m, y {worst[1]:.1e} m, psi {worst[2]:.1e} rad, v {worst[3]:.1e} m/s")
assert worst[0] < 1e-6 and worst[1] < 1e-6
assert worst[2] < 1e-9 and worst[3] < 1e-9

### Braking to a stop

The race windows never stop the car, so check the one place where the model is not smooth.
Under the brake, v' jumps at v = 0 (the brake acts against the motion and never drives the car backwards).
One RK4 step across that jump is wrong, so `predict` integrates a tick in which the brake can stop the car in substeps that hold v at 0, as the car does.
Compare it with racekit's car itself, from low speeds, for 1 s of braking actions.

In [ ]:
worst_stop = 0.0
for v0 in (0.0, 0.01, 0.2, 1.0, 3.0, 8.0):
    for actions in ([(0.0, 0.1, 1.0)] * 20, [(0.2, -0.1, 0.3)] * 20, [(0.3, 0.0, 1.0)] * 20,
                    [(0.0, 0.0, 1.0)] * 5 + [(0.5, 0.05, 0.0)] * 15):
        car = KinematicCar()
        car.reset(1.0, 2.0, 0.3, v0)
        for throttle, steer, brake in actions:
            car.advance(throttle, steer, brake, 0.0, 0.0, CONTROL_DT, 10)
        px, py, _, pv = predict((1.0, 2.0, 0.3, v0), actions)
        worst_stop = max(worst_stop, abs(px - car.x), abs(py - car.y), abs(pv - car.v))
        assert pv >= 0.0
print(f"braking to a stop, 24 cases: max error {worst_stop:.1e}")
assert worst_stop < 1e-6

## The model, pinned

| | Euler, 1 step per tick | Euler, 10 substeps | RK4, 1 step per tick |
|---|---|---|---|
| steady circle, max \|dist − R\| | 0.25 m (h·v/2, bounded) | | < 1e-6 m |
| 1 s prediction from telemetry, max | 0.378 m | 0.038 m | < 1e-6 m |
| error per halving of h | × 1/2 (order 1) | | × 1/16 (order 4) |

At the race's 0.05 s tick, RK4's integration error is negligible: from here on, the doubt belongs to the model, not to the integrator.